# Ingesta de Datos y Variables Climáticas

Este notebook orquesta la preparación de los datos base. Lee directamente el histórico en Excel/CSV mediante `ingesta_precios.py`, ejecuta el `preprocesamiento.py` y finaliza uniendo el histórico del clima usando `integrar_clima.py`.

In [3]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Importar los modulos de tu arquitectura
from ingesta_precios import procesar_archivo_precios
from preprocesamiento import preprocesar_datos

## 1. Ingesta del Archivo Excel

Simulamos el objeto `FileStorage` de Flask para poder reutilizar el mismo script de tu aplicación web sin modificarle ni una línea.

In [4]:
# Clase para simular el comportamiento de Flask FileStorage en Jupyter
class FileStorageMock:
    def __init__(self, filepath):
        self.filename = os.path.basename(filepath)
        self.stream = open(filepath, 'rb')
        
    def read(self, *args, **kwargs):
        return self.stream.read(*args, **kwargs)
        
    def seek(self, *args, **kwargs):
        return self.stream.seek(*args, **kwargs)
        
    def tell(self):
        # Pandas necesita este método para saber la posición actual
        return self.stream.tell()
        
    def seekable(self):
        # Pandas verifica si el flujo de datos permite saltos (seek)
        return self.stream.seekable()
        
    def close(self):
        self.stream.close()

# Rutas de los archivos
EXCEL_PATH = "data/precios-mercados-mayoristas-bodegas-comerciales.xlsx"
CSV_CRUDO_PATH = "data/processed/dataset_crudo_sipa.csv"

print(f"Ingiriendo datos desde: {EXCEL_PATH}")

try:
    import os
    from ingesta_precios import procesar_archivo_precios
    
    archivo_mock = FileStorageMock(EXCEL_PATH)
    # Llamamos a tu script modular
    df_crudo, reporte = procesar_archivo_precios(archivo_mock, CSV_CRUDO_PATH)
    archivo_mock.close()
    
    print("\n=== REPORTE DE INGESTA ===")
    print(f"Registros procesados: {reporte['registros_completos']}")
    print(f"Quincenas detectadas: {reporte['quincenas']}")
    print(f"Archivo guardado en:  {CSV_CRUDO_PATH}")
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo Excel en {EXCEL_PATH}")
    print("Asegúrate de que el archivo esté en la carpeta 'data/'.")

Ingiriendo datos desde: data/precios-mercados-mayoristas-bodegas-comerciales.xlsx

=== REPORTE DE INGESTA ===
Registros procesados: 57284
Quincenas detectadas: 174
Archivo guardado en:  data/processed/dataset_crudo_sipa.csv


## 2. Preprocesamiento Base

Ejecutamos la limpieza, generación de variables continuas (lags, promedios móviles) y codificación de clases.

In [5]:
if os.path.exists(CSV_CRUDO_PATH):
    print("Cargando dataset crudo para preprocesamiento...")
    df_base = pd.read_csv(CSV_CRUDO_PATH)
    
    resultado_preproc = preprocesar_datos(df_base)
    df_preprocesado = resultado_preproc['dataset_final']
    
    # Sobrescribimos el archivo para que el script de clima lo pueda leer
    PREPROC_PATH = "data/processed/dataset_preprocesado_sipa.csv"
    df_preprocesado.to_csv(PREPROC_PATH, index=False, encoding='utf-8-sig')
    
    print(f"\nPreprocesamiento exitoso. {len(df_preprocesado)} registros generados.")
else:
    print("No se puede preprocesar porque no se genero el dataset crudo.")

Cargando dataset crudo para preprocesamiento...

Preprocesamiento exitoso. 50862 registros generados.


## 3. Integración de Clima Histórico

Utilizamos el script `integrar_clima.py` que inyecta los datos meteorológicos (temperatura, precipitaciones y rezagos climáticos) mediante un left join por provincia, año y mes.

In [6]:
print("Ejecutando script de integracion de clima...\n")

# El comando magico %run ejecuta el script .py exactamente igual que si lo corrieras en la terminal
%run "integrar_clima.py"


Ejecutando script de integracion de clima...

Clima: 2975 filas, provincias: ['AZUAY', 'BOLIVAR', 'CANAR', 'CARCHI', 'CHIMBORAZO', 'COTOPAXI', 'EL ORO', 'GUAYAS', 'IMBABURA', 'LOJA', 'LOS RIOS', 'MANABI', 'PERU', 'PICHINCHA', 'SANTA ELENA', 'SANTO DOMINGO DE LOS TSACHILAS', 'TUNGURAHUA']
Preprocesado: 50862 filas
Se eliminaron columnas climáticas previas: ['clima_temp_media', 'clima_precipitacion_mm', 'clima_temp_lag1', 'clima_temp_lag2', 'clima_precip_lag1', 'clima_precip_lag2', 'clima_precip_acc3']

Cobertura clima: 100.0% de filas tienen datos climáticos.
Dataset final: 50862 filas, 41 columnas
Columnas climáticas: clima_temp_media, clima_precipitacion_mm, clima_temp_lag1, clima_temp_lag2, clima_precip_lag1, clima_precip_lag2
Guardado en: c:\Users\edu-g\Documents\GitHub\Clasificador-Precios-SIPA\data\processed\dataset_preprocesado_sipa.csv


## 4. Visualización del Dataset Final

Este es el DataFrame listo para pasar a la fase de Entrenamiento.

In [7]:
FINAL_PATH = "data/processed/dataset_preprocesado_sipa.csv"

if os.path.exists(FINAL_PATH):
    df_final = pd.read_csv(FINAL_PATH)
    display(df_final.head(10))
    
    print(f"\nDimensiones finales: {df_final.shape}")
    print("Columnas del clima integradas:")
    cols_clima = [c for c in df_final.columns if 'clima' in c]
    print(cols_clima)
else:
    print("Hubo un error en los pasos anteriores.")

,producto,provincia,canton,mercado,presentacion,tipo_mercado,periodo,precio_actual,precio_t1,precio_t2,...,mercado_encoded,presentacion_encoded,tipo_mercado_encoded,categoria_perecedero,clima_temp_media,clima_precipitacion_mm,clima_temp_lag1,clima_temp_lag2,clima_precip_lag1,clima_precip_lag2
0,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-02,30.500000,28.500000,NaN,...,0.820395,0.693245,1.015672,1,13.458621,271.4,13.812903,13.634159,256.2,144.35
1,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-03,39.333333,30.500000,28.500000,...,0.820395,0.693245,1.015672,1,13.193548,177.2,13.458621,13.812903,271.4,256.20
2,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-04,40.000000,39.333333,30.500000,...,0.820395,0.693245,1.015672,1,13.783333,151.2,13.193548,13.458621,177.2,271.40
3,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-05,36.000000,40.000000,39.333333,...,0.820395,0.693245,1.015672,1,12.806452,92.5,13.783333,13.193548,151.2,177.20
4,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-06,35.571429,36.000000,40.000000,...,0.820395,0.693245,1.015672,1,12.283333,58.7,12.806452,13.783333,92.5,151.20
5,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-07,39.500000,35.571429,36.000000,...,0.820395,0.693245,1.015672,1,11.667742,68.1,12.283333,12.806452,58.7,92.50
6,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-08,43.250000,39.500000,35.571429,...,0.820395,0.693245,1.015672,1,11.996774,88.4,11.667742,12.283333,68.1,58.70
7,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-09,43.250000,43.250000,39.500000,...,0.820395,0.693245,1.015672,1,12.483333,76.4,11.996774,11.667742,88.4,68.10
8,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-10,43.250000,43.250000,43.250000,...,0.820395,0.693245,1.015672,1,13.919355,176.4,12.483333,11.996774,76.4,88.40
9,Aguacate Fuerte,AZUAY,Cuenca,Cuenca - EL ARENAL,Ciento,Mayorista,2012-11,43.250000,43.250000,43.250000,...,0.820395,0.693245,1.015672,1,14.603333,235.2,13.919355,12.483333,176.4,76.40



Dimensiones finales: (50862, 41)
Columnas del clima integradas:
['clima_temp_media', 'clima_precipitacion_mm', 'clima_temp_lag1', 'clima_temp_lag2', 'clima_precip_lag1', 'clima_precip_lag2']
